In [4]:
import numpy as np
import json

# Set random seed for reproducibility
np.random.seed(42)

# extract each data["input"] from ./amazon_data/CDs_and_Vinyl_train.json, and then get the text between "The user has played the following musics before: " and ", please write a new musics that the user may bought"

'''
The structure of .json is like:

{
        "instruction": "Given a list of musics the user recently enjoys, please recommend a new musics that the user may like",
        "input": "The user has played the following musics before: \"Decade\", \"Comes a Time\", \"The Andy Griffith Show\", \"Country Strong Soundtrack\", \"The Best Of The Grateful Dead\", \"Revenge\", \"Picking up the Pieces\", \"The Division Bell\", please write a new musics that the user may bought",
        "output": "\"Wish You Were Here\"\n"
    },
'''

interactions_sampled = []
interactions_history = []

with open('./amazon_data/CDs_and_Vinyl_train.json', 'r') as f:
    json_content = f.read()

    data_list = json.loads(json_content)
    print("Length of JSON array: ", len(data_list))

    # randomly sample 4096 integers between 0 and len(data_list)
    sampled_indices = np.random.choice(len(data_list), size=4096, replace=False)
    sampled_data = [data_list[i] for i in sampled_indices]

    # get the rest out of sampled_data
    rest_data = [data for i, data in enumerate(data_list) if i not in sampled_indices]

    for data in rest_data:
        if "input" in data:
            input_text = data["input"]
            start = input_text.find("The user has played the following musics before: ") + len("The user has played the following musics before: ")
            end = input_text.find(", please write a new musics that the user may bought")
            if start != -1 and end != -1:
                interactions_history.append("A user played the following musics before in sequence: " + input_text[start:end])


print(f"Total interactions: {len(data_list)}")

print(interactions_history[0])

# save sampled_data to ./amazon_data/CDs_and_Vinyl_train_sampled.json
with open('./amazon_data/CDs_and_Vinyl_train_sampled_orig.json', 'w') as f:
    # convert to json array
    json_array = [data for data in sampled_data]
    json.dump(json_array, f)

with open('./amazon_data/CDs_and_Vinyl_train_history.json', 'w') as f:
    # convert to json array
    json_array = [data for data in interactions_history]
    json.dump(json_array, f)

Length of JSON array:  76287
Total interactions: 76287
A user played the following musics before in sequence: "Christmas Eve and Other Stories", "Thriller", "And Then There Were Three", "A Trick of the Tail", "Christmas Attic, The"


In [12]:
# read and store the unique titles from CDs_and_Vinyl_test.json, CDs_and_Vinyl_train.json, and CDs_and_Vinyl_valid.json
# the titles are in data["input"] and data["output"], which are always between \" and \", there could be multiple titles in data["input"]

import re
import os

titles = set()

# Function to extract titles from text (titles are enclosed in double quotes)
def extract_titles(text):
    # Find all text between quotes
    pattern = r'"([^"]*)"'
    return re.findall(pattern, text)

# Process each JSON file
file_paths = [
    './amazon_data/CDs_and_Vinyl_train.json',
    './amazon_data/CDs_and_Vinyl_test.json',
    './amazon_data/CDs_and_Vinyl_valid.json'
]


for file_path in file_paths:
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
        
    try:
        with open(file_path, 'r') as f:
            json_content = f.read()
        
        data_list = json.loads(json_content)
        print(f"  Found {len(data_list)} entries")
        
        for data in data_list:
            # Extract titles from input
            if "input" in data:
                input_titles = extract_titles(data["input"])
                for title in input_titles:
                    titles.add(title)
            
            # Extract title from output
            if "output" in data:
                output_titles = extract_titles(data["output"])
                for title in output_titles:
                    titles.add(title)
    
    except Exception as e:
        print(f"  Error processing {file_path}: {str(e)}")

print(f"Total unique titles found: {len(titles)}")

# Save all titles to a file
with open('./amazon_data/unique_titles.txt', 'w') as f:
    for title in sorted(titles):
        f.write(f"{title}\n")

print(list(titles)[:5])

  Found 76287 entries
  Found 11756 entries
  Found 12424 entries
Total unique titles found: 13111
['Jimi Hendrix: Live at the Fillmore East', 'Total Fucking Darkness', 'Distant Satellites', 'Exactly Like This', 'To The King Vertical Worship']


In [ ]:
# for each title, find the corresponding entry from meta_CDs_and_Vinyl.json accoring to value of ["title"], then get the key ["price"], ["salesRank"], ["brand"], ["categories"]

import json
from tqdm.notebook import tqdm
import pandas as pd

# Load the unique titles we found earlier
with open('./amazon_data/unique_titles.txt', 'r') as f:
    unique_titles = [line.strip() for line in f]
print(f"Loaded {len(unique_titles)} unique titles")


# Load metadata file
meta_data = []
try:
    with open('./amazon_data/meta_CDs_and_Vinyl.json', 'r') as f:
        # The metadata file could be one JSON per line or a single JSON array
        first_char = f.read(1)
        f.seek(0)  # Reset file pointer
        
        if first_char == '[':
            # It's a JSON array
            meta_data = json.load(f)
        else:
            # It's JSON Lines format (one JSON object per line)
            for line in f:
                line = line.strip()
                if line:
                    try:
                        item = json.loads(line)
                        meta_data.append(item)
                    except json.JSONDecodeError:
                        continue
    
    print(f"Loaded metadata for {len(meta_data)} items")
except Exception as e:
    print(f"Error loading metadata file: {str(e)}")

# Create a dictionary for fast lookup by title
print("Building title lookup index...")
title_to_meta = {}
for item in meta_data:
    if "title" in item:
        title = item["title"]
        title_to_meta[title] = item

# Try to match our titles with metadata
print("Matching titles with metadata...")
matched_data = []

for title in unique_titles:
    if title in title_to_meta:
        # Direct match
        meta_item = title_to_meta[title]
        match_type = "exact"
    else:
        # Try fuzzy matching - look for titles containing our title as a substring
        match_type = "fuzzy"
        meta_item = None
        for meta_title, item in title_to_meta.items():
            if title in meta_title or meta_title in title:
                meta_item = item
                break
    
    if meta_item:
        # Extract the requested fields
        price = meta_item.get("price", None)
        sales_rank = meta_item.get("salesRank", None)
        brand = meta_item.get("brand", None)
        categories = meta_item.get("categories", [])
        
        matched_data.append({
            "title": title,
            "price": price,
            "salesRank": sales_rank,
            "brand": brand,
            "categories": categories,
            "match_type": match_type
        })

# Convert to DataFrame for easier analysis
df = pd.DataFrame(matched_data)

# Print summary
print(f"\nMatched {len(matched_data)} out of {len(unique_titles)} titles ({len(matched_data)/len(unique_titles)*100:.2f}%)")

# Show sample of the data
print("\nSample of matched data:")
if len(df) > 0:
    print(df.head(2).to_string())
    
    # Also save as JSON for later use
    with open('./amazon_data/matched_titles_metadata.json', 'w') as f:
        json.dump(matched_data, f)
    print("Saved matched data to './amazon_data/matched_titles_metadata.json'")
else:
    print("No matches found!")

Loaded 13111 unique titles
Loaded metadata for 516914 items
Building title lookup index...
Matching titles with metadata...

Matched 13111 out of 13111 titles (100.00%)

Sample of matched data:
             title   price salesRank              brand categories match_type
0  - Chicago XXXVI  $10.97      None  Richard X. Heyman         []      fuzzy
1               />  $39.98      None                            []      fuzzy
Saved matched data to './amazon_data/matched_titles_metadata.json'


In [3]:
# read ./amazon_data/matched_titles_metadata.json
with open('./amazon_data/matched_titles_metadata.json', 'r') as f:
    matched_titles_metadata = json.load(f)

# read ./amazon_data/CDs_and_Vinyl_train_history.json
with open('./amazon_data/CDs_and_Vinyl_train_history.json', 'r') as f:
    history_data = json.load(f)

corpora_dict = []

for i, interaction in enumerate(history_data):
    corpora_dict.append({"id": str(i), "contents": interaction})

start_id = len(history_data)
for meta in matched_titles_metadata:
    corpora_dict.append({"id": str(start_id), "contents": f"Title: {meta['title']}, Price: {meta['price']}, SalesRank: {meta['salesRank']}, Brand: {meta['brand']}, Categories: {', '.join(meta['categories']) if isinstance(meta['categories'], list) else meta['categories']}"})
    start_id += 1

# save corpora_dict to a file
with open('./amazon_data/corpora.jsonl', 'w') as f:
    for item in corpora_dict:
        f.write(json.dumps(item) + "\n")


In [7]:
import re

def extract_title(solution_str):
    pattern = r'(?s)<answer>(.*?)</answer>'
    match = re.search(pattern, solution_str)
    return match.group(1) if match else None

strinter = "For example, <answer> \"Revenge\"."
if extract_title(strinter):
    print(extract_title(strinter))